In [1]:
# Cell 1 — Install dependencies
!pip install -q torch transformers datasets peft accelerate bitsandbytes sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
Tesla T4, 15360 MiB


In [2]:
# Cell 2 — Imports
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch:  2.10.0+cu128
CUDA:     True
GPU:      Tesla T4
VRAM:     15.6 GB


In [3]:
# Cell 3 — Config
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LEN    = 512
BATCH_SIZE = 4
EPOCHS     = 3
LR         = 2e-4

print(f"Model: {MODEL_NAME}")
print(f"Max seq length: {MAX_LEN} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LR}")


Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Max seq length: 512 | Batch: 4 | Epochs: 3 | LR: 0.0002


In [4]:
# Cell 4 — 4-bit QLoRA config + load tokenizer & model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Memory footprint: 1.01 GB


In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [8]:
# Cell 6 — Upload train.jsonl and load dataset
from google.colab import files

print("Upload train.jsonl:")
files.upload()   # file lands at /content/train.jsonl

dataset = load_dataset(
    "json",
    data_files={"train": "/content/train.jsonl"},
)

print(f"\nLoaded: {len(dataset['train'])} samples")
print("Sample:", dataset["train"][0])


Upload train.jsonl:


Saving train.jsonl to train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]


Loaded: 1518 samples
Sample: {'instruction': 'What is the relationship between the use of levothyroxine and the improvement of symptoms in a patient diagnosed with subclinical hypothyroidism with hyperprolactinemia? (related to the task of Relation Extraction)', 'input': "Discharge Summary:\n\nPatient Name: [Patient Name]\nMedical Record Number: [MRN]\nDate of Service: [Date of Service]\n\nHISTORY OF PRESENT ILLNESS:\nThe patient is a 48-year-old female from Honduras who presented to the clinic with chief complaints of breast tenderness and galactorrhea for the past two to three weeks. She had no history of tobacco, marijuana, alcohol, illicit drugs, or over the counter medicines. Physical examination revealed non-tender, diffuse enlargement of the thyroid gland. On palpation, breast examination illustrated bilateral tenderness and milky yellowish discharge. Patient doesn't complain of headache or vision changes.\n\nMEDICAL HISTORY:\nThe patient had a past medical history of hypertens

In [9]:
# Cell 7 — Format and tokenize dataset
def format_example(example):
    inp_block = f"\n### Input:\n{example['input']}" if example.get("input", "").strip() else ""
    prompt = (
        f"### Instruction:\n{example['instruction']}"
        f"{inp_block}\n\n"
        f"### Response:\n{example['output']}"
    )
    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(
    format_example,
    remove_columns=dataset["train"].column_names,
)

print(f"Tokenized: {len(tokenized_dataset['train'])} samples")
print("Keys:", list(tokenized_dataset["train"][0].keys()))


Map:   0%|          | 0/1518 [00:00<?, ? examples/s]

Tokenized: 1518 samples
Keys: ['input_ids', 'attention_mask', 'labels']


In [10]:
# Cell 8 — Training arguments
training_args = TrainingArguments(
    output_dir="/content/qlora-output",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
)

print(training_args)


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,

In [11]:
# Cell 9 — Build Trainer and train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

train_result = trainer.train()
print(train_result.metrics)


Step,Training Loss
50,1.462840
100,1.388620
150,1.366566
200,1.362367
250,1.362243
300,1.340397
350,1.315302
400,1.296231
450,1.255890
500,1.225740


{'train_runtime': 2519.4314, 'train_samples_per_second': 1.808, 'train_steps_per_second': 0.452, 'total_flos': 1.4649204337016832e+16, 'train_loss': 1.2412538294206585, 'epoch': 3.0}


In [12]:
# Cell 10 — Quick inference test
model.eval()

test_prompts = [
    "What is gradient descent in machine learning?",
    "Explain the difference between SQL and NoSQL databases.",
    "What are the benefits of using Docker containers?",
]

for prompt in test_prompts:
    inputs = tokenizer(
        f"<|user|>\n{prompt}</s>\n<|assistant|>\n",
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=False)
    answer  = decoded.split("<|assistant|>")[-1].replace("</s>", "").strip()
    print(f"Q: {prompt}")
    print(f"A: {answer}")
    print("-" * 70)

Q: What is gradient descent in machine learning?
A: in response to the need to solve non-linear problems in machine learning algorithms.

In summary, gradient descent is an optimization technique used for training ANNs and other deep learning models to find the minimum or maximum value of a function using backpropagation.
----------------------------------------------------------------------
Q: Explain the difference between SQL and NoSQL databases.
A: SQL (Structured Query Language) is a standardized query language used for managing data in relational databases, while NoSQL (Not Only SQL) databases are not limited to relational databases and can handle unstructured data, non-relational data, or data that doesn't fit into a traditional schema.

In SQL, the query process involves retrieving and manipulating data based on predetermined criteria using a specific set of rules. In contrast, in NoSQL databases, queries are often more flexible and adaptable to different types of data, allowin

In [13]:
# Cell 11 — Save the fine-tuned LoRA adapters

import os

ADAPTER_DIR = "/content/adapters"
os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR, safe_serialization=False)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Training complete. Adapter saved to", ADAPTER_DIR)
for fname in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, fname))
    print(f"  {fname}  ({sz / 1024:.1f} KB)")

# Confirm adapter_model.bin exists
assert "adapter_model.bin" in os.listdir(ADAPTER_DIR), "adapter_model.bin not found!"
print("\nadapter_model.bin confirmed.")


Training complete. Adapter saved to /content/adapters
  README.md  (5.1 KB)
  adapter_config.json  (1.0 KB)
  adapter_model.bin  (49393.7 KB)
  chat_template.jinja  (0.4 KB)
  tokenizer.json  (3534.2 KB)
  tokenizer_config.json  (0.4 KB)

adapter_model.bin confirmed.
